# 08 · CFTR2 — the disease-specific *functional* truth set

[CFTR2](https://cftr2.org) classifies *CFTR* variants using **patient outcomes + in-vitro CFTR function assays** — a different, more *functional* kind of evidence than ClinVar's clinical assertions. That functional axis makes it an **orthogonal** truth set (notebook 12). The **REAL** full CFTR2 release is built locally (gitignored); a fresh clone falls back to DEMO.

> ✅ **REAL data** — the full public CFTR2 list (30 January 2026 release, ~2,097 variants), built locally into `data/cftr2_2026-01-30.csv`. Redistributed under cftr2.org's public data-use terms; please cite CFTR2 if you use it.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 5 · CFTR2 — a disease-specific, *functional* reference

[CFTR2](https://cftr2.org) is different in kind from ClinVar. It is a **CF-specific** database that classifies *CFTR* variants as:

- **CF-causing**
- **Varying clinical consequence** (formerly "CF-causing (mild)")
- **Non CF-causing**
- **No interpretation available** — not yet enough evidence

Crucially, CFTR2's calls are built from **two kinds of evidence together**:
1. **Patient data** — real clinical outcomes across thousands of people with CF who carry the variant.
2. **In-vitro CFTR function assays** — measuring, in the lab, how much working chloride-channel the variant protein actually produces.

That second, functional axis makes CFTR2 **partially orthogonal** to ClinVar — but *not* independent of it (the two share clinical evidence and cross-cite; see notebook 12).

In [2]:
cftr2 = tk.load_cftr2()          # REAL — full CFTR2 30 January 2026 release
print('source :', cftr2['source'].unique(), '| variants:', len(cftr2),
      '| with a missense key:', (cftr2['protein_variant'] != '').sum())
print()
print(cftr2['cftr2_class'].value_counts().to_string())
cftr2.head(8)

source : ['REAL'] | variants: 2097 | with a missense key: 780

cftr2_class
CF-causing                      1245
No interpretation available      722
Varying clinical consequence      83
Non CF-causing                    42


,protein_variant,legacy_name,protein_name,cdna_name,cftr2_alleles,cftr2_af,cftr2_class,grch38_chr,grch38_pos,grch38_ref,grch38_alt,source
0,,F508del,p.Phe508del,c.1521_1523del,137363,0.650682595473364,CF-causing,7.0,117559590.0,ATCT,A,REAL
1,,G542X,p.Gly542X,c.1624G>T,5752,0.027246975453089916,CF-causing,7.0,117587778.0,G,T,REAL
2,G551D,G551D,p.Gly551Asp,c.1652G>A,3831,0.01814728146049852,CF-causing,7.0,117587806.0,G,A,REAL
3,N1303K,N1303K,p.Asn1303Lys,c.3909C>G,3551,0.01682093355944407,CF-causing,7.0,117652877.0,C,G,REAL
4,,W1282X,p.Trp1282X,c.3845G>A|c.3846G>A,2500,0.011842391973700416,CF-causing,NaN,NaN,NaN,NaN,REAL
5,R117H,R117H,p.Arg117His,c.350G>A,2262,0.010714996257804137,Varying clinical consequence,7.0,117530975.0,G,A,REAL
6,,3849+10kbC->T,p.?,c.3718-2477C>T,1990,0.00942654401106553,CF-causing,7.0,117639961.0,C,T,REAL
7,,621+1G->T,p.?,c.489+1G>T,1860,0.008810739628433109,CF-causing,7.0,117531115.0,G,T,REAL


### Which variants get a `protein_variant` key — and what are the rest?

The 1-letter `protein_variant` key (e.g. `G551D`) is derived by `build_cftr2.py` from
the protein name with a regex for *simple single-residue missense* only. It exists for
~780 of ~2,097 variants. **The other ~1,317 are NOT all splice variants** — most are
deletions and nonsense. They carry an empty key and must be joined by `cdna_name` or
genomic coordinates instead.

In [3]:
import re
cf = cftr2.copy()
cf["has_key"] = cf["protein_variant"].fillna("") != ""
print("with missense key:", int(cf["has_key"].sum()), "| without:", int((~cf["has_key"]).sum()))

def category(row):
    p, c = str(row.get("protein_name") or ""), str(row.get("cdna_name") or "")
    if "del" in p or "del" in c: return "deletion/indel"
    if "X" in p or "Ter" in p:   return "nonsense (stop-gain)"
    if "ins" in c or "dup" in c: return "insertion/dup"
    if ("+" in c) or ("-" in c and "c." in c): return "splice/intronic"
    if "=" in p: return "synonymous"
    return "other/complex"

nokey = cf[~cf["has_key"]]
print("\nWhat the NON-missense (no-key) variants actually are:")
print(nokey.apply(category, axis=1).value_counts().to_string())

with missense key: 780 | without: 1317

What the NON-missense (no-key) variants actually are:
deletion/indel          540
nonsense (stop-gain)    347
splice/intronic         291
other/complex            50
insertion/dup            48
synonymous               41


> ✅ **This is the REAL CFTR2 list** — the full public 30 January 2026 release (~2,097 variants), built locally into `data/cftr2_2026-01-30.csv` and built from the cftr2.org download by `build_cftr2.py`. A 1-letter `protein_variant` key (e.g. `G551D`) is derived for the ~780 simple-missense variants so CFTR2 class joins straight onto the AlphaMissense / gnomAD tables; non-missense rows keep their legacy / cDNA names and GRCh38 coordinates. CFTR2 data is redistributed under cftr2.org's public data-use terms — please cite CFTR2 if you use it.

> **Provenance & pinning.** The list comes from cftr2.org's **variant-list *history* tab**, which exposes dated, versioned releases — pin the exact release (**30 Jan 2026**) in `data_manifest.json` so the analysis stays reproducible and older releases remain re-retrievable. CFTR2 classification methodology: Sosnay et al. 2013, *Nat Genet*, PMID 23974870.

## 6 · ★ Key teaching point: how *orthogonal* is CFTR2, really?

Here is the subtle trap this notebook is building toward.

Many predictors — especially **supervised** ones like **REVEL** — were *trained* using variant labels that ultimately trace back to clinical databases in the **ClinVar lineage**. So if you 'benchmark' REVEL against ClinVar, you may partly be testing it **on data related to what it learned from** — grading its own homework. This is **circularity** (label leakage).

**CFTR2 helps, but does not fully break the loop.** Its **functional-assay** component is a wet-lab measurement a sequence model never trained on — genuinely independent signal. *However*, CFTR2 is **not** an independent gold standard:

- its **patient/clinical** component overlaps the same evidence that feeds ClinVar;
- **ClinVar entries cite CFTR2**, and **CFTR2 informs the ACMG CFTR guidance** ClinVar submitters follow — the databases **cross-reference**.

So a variant's CFTR2 call and its ClinVar call are **correlated by construction**, not two independent opinions.

> **Rule of thumb:** use ClinVar for **breadth**; lean on CFTR2's *functional* measurements as **partial** orthogonal evidence — but never report "agrees with CFTR2" as if it were independent of ClinVar. Notebook 12 makes this precise.

## Example: the shared missense worked-example panel, scored by **CFTR2**

The same fixed panel of famous CFTR **missense** variants runs through every missense tool
(notebooks 01–08), so you can follow one set of variants across the series. The
variant list is `tk.A1_PANEL_VARIANTS` / `tk.A2_KNOWN_CDNA` (shared in `toolkit.py`); the
**scoring is shown inline below** so you can see exactly how CFTR2 is joined onto it.

> The assembled cross-tool table for all tools at once is `tk.a1_panel()()`.

In [4]:
panel = tk.A1_PANEL_VARIANTS
cf = tk.load_cftr2()
cf[cf['protein_variant'].isin(panel)][['protein_variant', 'cdna_name', 'cftr2_class']].reset_index(drop=True)

,protein_variant,cdna_name,cftr2_class
0,G551D,c.1652G>A,CF-causing
1,R117H,c.350G>A,Varying clinical consequence
2,D1152H,c.3454G>C,Varying clinical consequence
3,G85E,c.254G>A,CF-causing
4,R334W,c.1000C>T,CF-causing
5,M470V,c.1408A>G,Non CF-causing
6,V520F,c.1558G>T,CF-causing
7,P205S,c.613C>T,CF-causing
8,R668C,c.2002C>T,Non CF-causing
9,G970D,c.2909G>A,CF-causing


## Key takeaways

1. **CFTR2** calls combine **patient data + functional assays**. The functional axis is *partially* orthogonal to ClinVar — but CFTR2 cross-cites ClinVar, so it is **not** an independent gold standard (notebook 12).
2. REAL when built locally (~2,097 variants; **780** have a 1-letter missense key, 779 of which join to real AlphaMissense). The other ~1,317 are non-missense (see the breakdown above) and join by `cdna_name` / genomic coordinates.
3. The **CFTR2-vs-ClinVar agreement** cross-check lived in the archived integration notebook.

**Next:** notebook 09 — **SpliceAI**, the first splice predictor.